In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
root = Path('/content')
photos = root / 'ur5e_openvla_training_bundle_20261119.zip'
records = root / 'ur5e_pi05_sidecar_20261119.zip'
print('ZIP sizes:', photos.stat().st_size, records.stat().st_size)
assert photos.stat().st_size == 64657316, 'photo upload incomplete'
assert records.stat().st_size == 467901, 'records upload incomplete'
print('ZIP entries:', len(ZipFile(photos).namelist()), len(ZipFile(records).namelist()))
ZipFile(photos).extractall(root)
ZipFile(records).extractall(root)
report = json.loads((root / 'ur5e_pi05_data_20261119/verification.json').read_text())
print('verification:', report)

In [ ]:
import sys, torch, importlib.metadata as md; print('Python:', sys.version.split()[0], 'Torch:', torch.__version__, 'CUDA:', torch.cuda.get_device_name(0)); print('LeRobot:', md.version('lerobot') if 'lerobot' in {x.metadata['Name'] for x in md.distributions()} else 'not installed')

In [ ]:
%pip install -q 'lerobot[training,pi]'

In [ ]:
%run /content/convert_ur5e_pi05_lerobot.py --records /content/ur5e_pi05_data_20261119 --images /content/ur5e_openvla_data --output /content/ur5e_pi05_lerobot

In [ ]:
import importlib.metadata as md; from huggingface_hub import hf_hub_download; print('LeRobot:', md.version('lerobot')); print('PaliGemma file:', hf_hub_download('google/paligemma-3b-pt-224', 'tokenizer_config.json'))

In [ ]:
from huggingface_hub import login; login()

In [ ]:
from huggingface_hub import hf_hub_download; print('PALIGEMMA_ACCESS_OK', hf_hub_download('google/paligemma-3b-pt-224', 'tokenizer_config.json'))

In [ ]:
!lerobot-train --help 2>&1 | tail -80

In [ ]:
!lerobot-train --dataset.repo_id=local/ur5e_pi05_train --dataset.root=/content/ur5e_pi05_lerobot/train --policy.type=pi05 --policy.pretrained_path=lerobot/pi05_base --policy.n_action_steps=10 --policy.empty_cameras=2 --policy.freeze_vision_encoder=true --policy.train_expert_only=true --policy.gradient_checkpointing=true --policy.dtype=bfloat16 --policy.device=cuda --policy.push_to_hub=false --output_dir=/content/ur5e_pi05_outputs/smoke --job_name=ur5e_pi05_smoke --batch_size=1 --num_workers=0 --steps=2 --save_freq=2 --log_freq=1 --seed=7

In [ ]:
import subprocess

cmd = [
    "lerobot-train",
    "--dataset.repo_id=local/ur5e_pi05_train",
    "--dataset.root=/content/ur5e_pi05_lerobot/train",
    "--policy.type=pi05",
    "--policy.pretrained_path=lerobot/pi05_base",
    "--policy.n_action_steps=10",
    "--policy.empty_cameras=2",
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    "--policy.gradient_checkpointing=true",
    "--policy.dtype=bfloat16",
    "--policy.device=cuda",
    "--policy.push_to_hub=false",
    "--output_dir=/content/ur5e_pi05_outputs/full_1epoch_seed7",
    "--job_name=ur5e_pi05_40demo_1epoch",
    "--batch_size=1",
    "--num_workers=0",
    "--steps=3600",
    "--save_freq=3600",
    "--log_freq=100",
    "--seed=7",
]
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path
root = Path("/content/ur5e_pi05_outputs/full_1epoch_seed7")
print("OUTPUT_EXISTS", root.exists())
for p in sorted(root.rglob("*")):
    if p.is_file():
        print(p.relative_to(root), round(p.stat().st_size / 1024**2, 2), "MiB")

In [ ]:
import json
from pathlib import Path
base = Path("/content/ur5e_pi05_outputs/full_1epoch_seed7/checkpoints/003600")
for name in ["training_state/training_step.json", "pretrained_model/train_config.json"]:
    data = json.loads((base / name).read_text())
    print(name, data if "training_step" in name else {k: data.get(k) for k in ["steps", "seed", "batch_size", "dataset", "policy"]})
print("DRIVE_MOUNTED", Path("/content/drive/MyDrive").exists())

In [ ]:
import inspect
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from lerobot.datasets.lerobot_dataset import LeRobotDataset
print("LOAD", inspect.signature(PI05Policy.from_pretrained))
print("ACTION", inspect.signature(PI05Policy.select_action))
ds = LeRobotDataset("local/ur5e_pi05_train", root="/content/ur5e_pi05_lerobot/train")
sample = ds[0]
print("SAMPLE", {k: tuple(v.shape) if hasattr(v, "shape") else str(v)[:100] for k, v in sample.items()})

In [ ]:
import subprocess, sys
code = """
import torch
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
path = "/content/ur5e_pi05_outputs/full_1epoch_seed7/checkpoints/003600/pretrained_model"
policy = PI05Policy.from_pretrained(path)
policy.to("cuda")
policy.eval()
print("RELOAD_OK", type(policy).__name__, "GPU", next(policy.parameters()).device, "PARAMS", sum(p.numel() for p in policy.parameters()), flush=True)
"""
result = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
print("EXIT_CODE", result.returncode)
print("STDOUT", result.stdout[-1500:])
print("STDERR_TAIL", result.stderr[-2500:])

In [ ]:
import inspect
from lerobot.policies import factory
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
print("FACTORY", [name for name in dir(factory) if "processor" in name])
print("SELECT_ACTION_SOURCE", inspect.getsource(PI05Policy.select_action)[:5000])
print("MAKE_SIGNATURE", inspect.signature(factory.make_pre_post_processors))

In [ ]:
import torch
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.datasets.lerobot_dataset import LeRobotDataset
path = "/content/ur5e_pi05_outputs/full_1epoch_seed7/checkpoints/003600/pretrained_model"
policy = PI05Policy.from_pretrained(path).to("cuda").eval()
pre, post = make_pre_post_processors(policy.config, pretrained_path=path)
val = LeRobotDataset("local/ur5e_pi05_validation", root="/content/ur5e_pi05_lerobot/validation")
sample = val[0]
batch = {k: (v.unsqueeze(0) if isinstance(v, torch.Tensor) else [v]) for k,v in sample.items() if k.startswith("observation.") or k == "task"}
print("RAW", {k: (tuple(v.shape) if isinstance(v, torch.Tensor) else v) for k,v in batch.items()}, flush=True)
batch = pre(batch)
print("PREPARED", {k: (tuple(v.shape) if isinstance(v, torch.Tensor) else str(v)[:60]) for k,v in batch.items()}, flush=True)
with torch.no_grad():
    action = post(policy.select_action(batch))
print("ACTION", action, "SHAPE", tuple(action.shape), "FINITE", bool(torch.isfinite(action).all()), flush=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import hashlib, os, shutil, time

source = Path('/content/ur5e_pi05_outputs/full_1epoch_seed7/checkpoints/003600/pretrained_model')
target = Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7/pretrained_model')
assert source.is_dir(), f'Missing checkpoint: {source}'
files = sorted(p for p in source.rglob('*') if p.is_file())
print('SOURCE_FILES', len(files), 'TOTAL_GiB', round(sum(p.stat().st_size for p in files) / 2**30, 3), flush=True)
target.mkdir(parents=True, exist_ok=True)
start = time.monotonic()
for src in files:
    dst = target / src.relative_to(source)
    dst.parent.mkdir(parents=True, exist_ok=True)
    size = src.stat().st_size
    if dst.exists() and dst.stat().st_size == size:
        print('ALREADY_PRESENT', src.relative_to(source), flush=True)
        continue
    if size > 100 * 2**20:
        partial = dst.with_name(dst.name + '.partial')
        copied = 0
        with src.open('rb') as fin, partial.open('wb') as fout:
            while chunk := fin.read(64 * 2**20):
                fout.write(chunk)
                copied += len(chunk)
                if copied % (1024 * 2**20) < len(chunk):
                    print('COPY_PROGRESS_GiB', round(copied / 2**30, 2), '/', round(size / 2**30, 2), flush=True)
        os.replace(partial, dst)
    else:
        shutil.copy2(src, dst)
    print('COPIED', src.relative_to(source), 'bytes', size, flush=True)
assert all((target / p.relative_to(source)).stat().st_size == p.stat().st_size for p in files)
print('DRIVE_COPY_SIZE_VERIFIED', target, 'files', len(files), 'seconds', round(time.monotonic()-start, 1), flush=True)


In [ ]:
from pathlib import Path
import importlib.metadata as md
root = Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7/pretrained_model')
print('checkpoint', root.exists(), (root/'model.safetensors').stat().st_size if (root/'model.safetensors').exists() else None)
print('train_data', Path('/content/ur5e_pi05_lerobot/train').exists())
print('drive_mounted', Path('/content/drive/MyDrive').exists())
for name in ('lerobot','torch','transformers','torchcodec'):
    try: print(name, md.version(name))
    except md.PackageNotFoundError: print(name, 'missing')


In [ ]:
import inspect
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
import lerobot.policies.factory as factory
from lerobot.datasets.lerobot_dataset import LeRobotDataset
print('load_signature', inspect.signature(PI05Policy.from_pretrained))
print('select_signature', inspect.signature(PI05Policy.select_action))
print('processor_helpers', [n for n in dir(factory) if 'processor' in n.lower()])
ds = LeRobotDataset('local/ur5e_pi05_train', root='/content/ur5e_pi05_lerobot/train')
item = ds[0]
print('sample_keys', list(item.keys()))
for k,v in item.items():
    if k in ('observation.state','action') or 'image' in k or k == 'task': print(k, getattr(v,'shape',None), str(v)[:120] if k=='task' else '')


In [ ]:
import inspect
from lerobot.policies.factory import make_pre_post_processors, load_pretrained_policy_processors
print('make_sig', inspect.signature(make_pre_post_processors))
print('load_sig', inspect.signature(load_pretrained_policy_processors))
print('load_source', inspect.getsource(load_pretrained_policy_processors)[:6500])
print('policy_select_source', inspect.getsource(PI05Policy.select_action)[:6500])


In [ ]:
from pathlib import Path
import json, inspect
base=Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7/pretrained_model')
for f in ('config.json','policy_preprocessor.json','policy_postprocessor.json'):
    obj=json.loads((base/f).read_text())
    print(f, str(obj)[:5000])
print('validation_exists',Path('/content/ur5e_pi05_lerobot/validation').exists())


In [ ]:
from pathlib import Path
import numpy as np, subprocess, sys, textwrap
from lerobot.datasets.lerobot_dataset import LeRobotDataset

run = Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7')
sample = LeRobotDataset('local/ur5e_pi05_validation', root='/content/ur5e_pi05_lerobot/validation')[0]
np.savez_compressed(run/'validation_scene_0_input.npz',
    image=sample['observation.images.image'].numpy(),
    state=sample['observation.state'].numpy(),
    task=np.array(sample['task']))
script = run/'verify_fresh_action.py'
script.write_text(textwrap.dedent("""
from pathlib import Path
import json, numpy as np, torch
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from lerobot.policies.factory import load_pretrained_policy_processors

run = Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7')
checkpoint = run/'pretrained_model'
data = np.load(run/'validation_scene_0_input.npz', allow_pickle=False)
observation = {
    'observation.images.image': torch.from_numpy(data['image']).float(),
    'observation.state': torch.from_numpy(data['state']).float(),
    'task': str(data['task'].item()),
}
assert observation['observation.images.image'].shape == (3, 256, 256)
assert observation['observation.state'].shape == (7,)
assert torch.cuda.is_available()
torch.manual_seed(7)
torch.cuda.manual_seed_all(7)
preprocess, postprocess = load_pretrained_policy_processors(checkpoint)
policy = PI05Policy.from_pretrained(checkpoint, local_files_only=True)
policy.to(device='cuda', dtype=torch.bfloat16)
policy.eval()
with torch.inference_mode():
    model_input = preprocess(observation)
    raw_action = policy.select_action(model_input)
    action = postprocess(raw_action).detach().cpu().float()
assert action.numel() == 7, (tuple(raw_action.shape), tuple(action.shape))
assert bool(torch.isfinite(action).all()), action
result = {
    'checkpoint': str(checkpoint),
    'input_file': str(run/'validation_scene_0_input.npz'),
    'task': observation['task'],
    'image_shape': list(observation['observation.images.image'].shape),
    'state_shape': list(observation['observation.state'].shape),
    'raw_action_shape': list(raw_action.shape),
    'output_shape': list(action.shape),
    'all_finite': True,
    'action': [round(float(x), 6) for x in action.reshape(-1)],
}
(run/'fresh_action_result.json').write_text(json.dumps(result, indent=2))
print('FRESH_PROCESS_ACTION_OK', json.dumps(result), flush=True)
"""), encoding='utf-8')
log = run/'verify_fresh_action.log'
with log.open('w', encoding='utf-8') as stream:
    process = subprocess.Popen([sys.executable, '-u', str(script)], stdout=stream, stderr=subprocess.STDOUT)
print('STARTED', process.pid, 'script', script, 'log', log)


In [ ]:
print('process_status', process.poll())
print('result_exists', (run/'fresh_action_result.json').exists())
print('log_tail', log.read_text()[-2500:])


In [ ]:
print('process_status', process.poll())
print('result_exists', (run/'fresh_action_result.json').exists())
print('log_tail', log.read_text()[-3500:])


In [ ]:
print('process_status', process.poll())
print('result_exists', (run/'fresh_action_result.json').exists())
print('log_tail', log.read_text()[-3500:])


In [ ]:
print('process_status', process.poll())
print('result_exists', (run/'fresh_action_result.json').exists())
print('log_tail', log.read_text()[-4500:])


In [ ]:
print('returncode', process.poll(), 'result', (run/'fresh_action_result.json').exists())
print(log.read_text()[-5500:])


In [ ]:
print('returncode', process.poll(), 'result', (run/'fresh_action_result.json').exists())
print(log.read_text()[-5500:])


In [ ]:
import subprocess
print(subprocess.run(['ps','-o','pid,stat,etime,%cpu,%mem,rss,wchan:24,cmd','-p',str(process.pid)],capture_output=True,text=True).stdout)
print(subprocess.run(['nvidia-smi','--query-compute-apps=pid,used_gpu_memory','--format=csv,noheader'],capture_output=True,text=True).stdout)


In [ ]:
print('returncode', process.poll(), 'result', (run/'fresh_action_result.json').exists())
print(log.read_text()[-7000:])


In [ ]:
old = "policy.to(device='cuda', dtype=torch.bfloat16)"
source = script.read_text()
assert source.count(old) == 1
script.write_text(source.replace(old, "policy.to(device='cuda')"), encoding='utf-8')
log.rename(run/'verify_fresh_action_first_dtype_error.log')
with log.open('w', encoding='utf-8') as stream:
    process = subprocess.Popen([sys.executable, '-u', str(script)], stdout=stream, stderr=subprocess.STDOUT)
print('RETRY_STARTED', process.pid, 'script', script)


In [ ]:
rc = process.wait(timeout=600)
print('exit_code', rc)
print('result_exists', (run/'fresh_action_result.json').exists())
print(log.read_text()[-6000:])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
import importlib.util, torch
p = Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7/pretrained_model')
print('GPU:', torch.cuda.get_device_name(0))
print('LeRobot:', importlib.util.find_spec('lerobot') is not None)
print('Drive:', Path('/content/drive/MyDrive').exists())
print('Checkpoint:', p.exists(), (p/'model.safetensors').exists())
print('MuJoCo:', importlib.util.find_spec('mujoco') is not None)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
import sys, json, torch
from pathlib import Path
p = Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7/pretrained_model')
print('Python', sys.version.split()[0], 'Torch', torch.__version__)
print('Checkpoint', p.exists(), 'weights GiB', round((p/'model.safetensors').stat().st_size/1024**3, 2))
config = json.loads((p/'config.json').read_text())
print('Config type', config.get('type'), 'action_steps', config.get('n_action_steps'))
print('Train config exists', (p/'train_config.json').exists())

In [ ]:
%pip install -q 'lerobot[training,pi]' mujoco

In [ ]:
import importlib.metadata as md
from pathlib import Path
for name in ['lerobot','mujoco','transformers','torch']:
    print(name, md.version(name))
root = Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7')
print('Files:', [p.name for p in root.iterdir() if p.is_file()])
print('Verify script:')
print((root/'verify_fresh_action.py').read_text()[:7000])

In [ ]:
%run /content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7/verify_fresh_action.py

In [ ]:
from pathlib import Path
import json
for name in ['/content', '/content/drive/MyDrive/VLA_experiments', '/content/ur5e_pi05_data_20261119']:
    p = Path(name)
    print('PATH', name, 'exists', p.exists())
    if p.exists(): print('CHILDREN', [x.name for x in list(p.iterdir())[:40]])
ckpt = Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7/pretrained_model')
config = json.loads((ckpt/'config.json').read_text())
for k in ['input_features','output_features','n_action_steps','chunk_size','device']:
    print('CONFIG', k, config.get(k))

In [ ]:
import subprocess, pathlib
root=pathlib.Path('/content/mujoco_menagerie')
if not root.exists():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','https://github.com/google-deepmind/mujoco_menagerie.git',str(root)],check=True)
    subprocess.run(['git','-C',str(root),'sparse-checkout','set','universal_robots_ur5e','robotiq_2f85'],check=True)
print('SCENE_ASSETS_READY', (root/'universal_robots_ur5e/scene.xml').exists(), (root/'robotiq_2f85/2f85.xml').exists())

In [ ]:
import os
os.environ['MUJOCO_GL']='egl'
import mujoco, numpy as np, torch
from pathlib import Path
scene=mujoco.MjSpec.from_file('/content/mujoco_menagerie/universal_robots_ur5e/scene.xml')
grip=mujoco.MjSpec.from_file('/content/mujoco_menagerie/robotiq_2f85/2f85.xml')
scene.option.cone=mujoco.mjtCone.mjCONE_ELLIPTIC
scene.option.impratio=10
scene.attach(grip,site=scene.site('attachment_site'),prefix='gripper/')
block=scene.worldbody.add_body(name='test_block',pos=[0.45,0,0.05])
block.add_freejoint(name='test_block_joint')
block.add_geom(name='block_geom',type=mujoco.mjtGeom.mjGEOM_BOX,size=[0.025,0.025,0.05],mass=0.05,rgba=[1,0.4,0.1,1],friction=[1.5,0.005,0.0001])
model=scene.compile(); sim=mujoco.MjData(model); mujoco.mj_resetDataKeyframe(model,sim,0)
block_j=mujoco.mj_name2id(model,mujoco.mjtObj.mjOBJ_JOINT,'test_block_joint'); block_addr=model.jnt_qposadr[block_j]
sim.qpos[block_addr:block_addr+7]=[0.4114,-0.0333,0.05,1,0,0,0]; mujoco.mj_forward(model,sim)
camera=mujoco.MjvCamera();camera.type=mujoco.mjtCamera.mjCAMERA_FREE;camera.lookat[:]=[0.2,0,0.28];camera.distance=1.8;camera.azimuth=130;camera.elevation=-35
renderer=mujoco.Renderer(model,height=256,width=256);renderer.update_scene(sim,camera=camera); fresh=renderer.render().copy()
run=Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7')
ref=np.load(run/'validation_scene_0_input.npz',allow_pickle=False); saved=ref['image']
print('MUJOCO',mujoco.__version__,'DT',model.opt.timestep,'MODEL',model.nq,model.nu)
print('SAVED',saved.shape,saved.min(),saved.max(),'FRESH',fresh.shape,fresh.min(),fresh.max())
print('STATE_DIFF',np.max(np.abs(ref['state']-np.r_[sim.qpos[:6],0.])))
ref_rgb=(np.transpose(saved,(1,2,0))*255 if saved.max()<=1.01 else np.transpose(saved,(1,2,0)))
print('IMAGE_MAE',float(np.abs(ref_rgb.astype(float)-fresh.astype(float)).mean()))
from PIL import Image
Image.fromarray(fresh).save('/content/pi05_ur5e_fresh_scene.png')

In [ ]:
print('POLICY_IN_MEMORY', 'policy' in globals(), 'PREPROCESS_IN_MEMORY', 'preprocess' in globals())
print('ACTUATORS',[(mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_ACTUATOR,i),model.actuator_ctrlrange[i].tolist()) for i in range(model.nu)])
print('POLICY_RESET',hasattr(policy,'reset'))
print('CURRENT_QPOS',sim.qpos[:6].tolist())

In [ ]:
import csv, json, time, imageio.v2 as imageio
from PIL import Image
out_dir=run/'rollout_pi05_state0_20261122'
out_dir.mkdir(exist_ok=True)
policy.reset(); torch.manual_seed(7); torch.cuda.manual_seed_all(7)
mujoco.mj_resetDataKeyframe(model,sim,0)
sim.qpos[block_addr:block_addr+7]=[0.4114,-0.0333,0.05,1,0,0,0]
mujoco.mj_forward(model,sim)
finger_j=mujoco.mj_name2id(model,mujoco.mjtObj.mjOBJ_JOINT,'gripper/right_driver_joint')
finger_addr=int(model.jnt_qposadr[finger_j])
finger_a=mujoco.mj_name2id(model,mujoco.mjtObj.mjOBJ_ACTUATOR,'gripper/fingers_actuator')
pinch=mujoco.mj_name2id(model,mujoco.mjtObj.mjOBJ_SITE,'gripper/pinch')
rows=[]; frames=[]; initial_z=float(sim.qpos[block_addr+2]); start_wall=time.perf_counter()
for step in range(90):
    renderer.update_scene(sim,camera=camera); rgb=renderer.render().copy(); frames.append(rgb)
    state=np.array([*sim.qpos[:6],sim.qpos[finger_addr]],dtype=np.float32)
    obs={'observation.images.image':torch.from_numpy(rgb.copy()).permute(2,0,1).float()/255,
         'observation.state':torch.from_numpy(state),'task':'pick up the orange block'}
    with torch.inference_mode():
        action=postprocess(policy.select_action(preprocess(obs))).detach().cpu().float().reshape(-1).numpy()
    if action.shape!=(7,) or not np.isfinite(action).all(): raise RuntimeError(f'Invalid action at {step}: {action}')
    ctrl=np.clip(action[:6],model.actuator_ctrlrange[:6,0],model.actuator_ctrlrange[:6,1])
    grip_cmd=0 if action[6]>=0 else 255
    sim.ctrl[:6]=ctrl;sim.ctrl[finger_a]=grip_cmd
    for _ in range(50): mujoco.mj_step(model,sim)
    z=float(sim.qpos[block_addr+2]); dist=float(np.linalg.norm(sim.site_xpos[pinch]-sim.qpos[block_addr:block_addr+3]))
    rows.append({'step':step,'time_s':round(float(sim.time),3),'block_z_m':z,'pinch_block_distance_m':dist,
                 'gripper_action':float(action[6]),'gripper_target':grip_cmd,
                 **{f'action_{i+1}':float(action[i]) for i in range(7)}})
    if step%10==0 or step==89:
        print('STEP',step+1,'BLOCK_Z',round(z,4),'DIST',round(dist,4),'GRIP','open' if grip_cmd==0 else 'close',flush=True)
        with (out_dir/'actions.csv').open('w',newline='') as f:
            w=csv.DictWriter(f,fieldnames=rows[0].keys());w.writeheader();w.writerows(rows)
renderer.update_scene(sim,camera=camera);frames.append(renderer.render().copy())
Image.fromarray(frames[0]).save(out_dir/'start.png'); Image.fromarray(frames[-1]).save(out_dir/'final.png')
imageio.mimsave(out_dir/'pi05_ur5e_state0.mp4',frames,fps=10)
summary={'steps':len(rows),'simulated_seconds':float(sim.time),'wall_seconds':time.perf_counter()-start_wall,
         'initial_block_z_m':initial_z,'peak_block_z_m':max(r['block_z_m'] for r in rows),
         'final_block_z_m':rows[-1]['block_z_m'],'lift_5cm':max(r['block_z_m'] for r in rows)-initial_z>=0.05,
         'gripper_close_steps':sum(r['gripper_target']==255 for r in rows),'output_dir':str(out_dir)}
(out_dir/'summary.json').write_text(json.dumps(summary,indent=2))
print('ROLLOUT_DONE',json.dumps(summary),flush=True)

In [ ]:
import csv, json, imageio.v2 as imageio
from pathlib import Path
p=Path('/content/drive/MyDrive/VLA_experiments/UR5e_Pi05_40demo_1epoch_seed7/rollout_pi05_state0_20261122')
with (p/'actions.csv').open() as f: saved_rows=list(csv.DictReader(f))
video=imageio.get_reader(str(p/'pi05_ur5e_state0.mp4')); first_video_frame=video.get_data(0);video.close()
print('VERIFIED',len(saved_rows),'CSV rows',len(json.loads((p/'summary.json').read_text())),'summary fields','video first frame',first_video_frame.shape)
print('FILES',[(x.name,x.stat().st_size) for x in p.iterdir()])
print('CLOSE_FIRST_STEP',next((int(r['step'])+1 for r in saved_rows if r['gripper_target']=='255'),None))
print('MIN_DISTANCE_M',min(float(r['pinch_block_distance_m']) for r in saved_rows))

In [ ]:
from IPython.display import display, Video
print('Pi0.5 UR5e rollout: 90 actions / 9 simulated seconds')
display(Video(str(p/'pi05_ur5e_state0.mp4'),embed=True))
close_row=next(r for r in saved_rows if r['gripper_target']=='255')
print('First close: step',int(close_row['step'])+1,'distance',round(float(close_row['pinch_block_distance_m']),3),'m')